# Automatic validation of the morphometric measurements

For each measurement (27), the image is classified as **measurable / non measurable** from
the keypoint confidences predicted by the pose model.

**Convention: the positive class is `non measurable`** (minority class, the one we care about).

**8 compared approaches:**

| key | features |
|---|---|
| `rule_conf_mean` | mean of the confidences of the direct kp of the measurement |
| `rule_conf_min` | minimum of the confidences of the direct kp of the measurement |
| `xgb_direct` / `rf_direct` | confidences of the direct kp + group one-hot |
| `xgb_related` / `rf_related` | confidences of the anatomical neighbourhood (`related_entities`) + group one-hot |
| `xgb_all` / `rf_all` | confidences of the 42 kp + group one-hot |

**Protocol:** stratified 5-fold, out-of-fold predictions. The decision threshold is chosen
on an inner partition of the *train* set (MCC maximisation), never on the test set.

**Cross-measurement comparison metric: the MCC** (robust to imbalance), complemented by the
normalised AP `(AP - prevalence) / (1 - prevalence)` which corrects the prevalence effect.

## 1. Configuration

In [ ]:
from __future__ import annotations

import gc
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_curve,
)
from xgboost import XGBClassifier
import joblib

from insect_anatomy import INSECT_GROUPS, MEAS_TO_KP, MEASUREMENTS, POINTS, related_entities
from dataset import STATUS_SUFFIX, build_dataset, label_report

warnings.filterwarnings("ignore")
matplotlib.use("Agg")  # no window: PNG files are written

# --- paths (to adapt) -------------------------------------------------------
DATA_DIR = Path("../../annotation/conf_classifier")                       # annotation CSV
DATABASE_DIR = Path("../../annotated_images/full databases")               # images/<group>/ tree
RESULTS_PATH = Path("../../pipeline/results_10.csv")  # pose model outputs
OUT_DIR = Path("outputs")

# --- protocol --------------------------------------------------------------
N_FOLDS = 5
RANDOM_STATE = 0
MIN_MINORITY = 20   # safeguard: measurement skipped below this number
NA_FILL = -1.0      # imputation sentinel for the random forest

for sub in ("pr_curves", "confusion"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)


def slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

## 2. Data loading

In [ ]:
frame, columns = build_dataset(DATA_DIR, DATABASE_DIR, RESULTS_PATH)
GROUP_COLS = [f"{g}_one_hot" for g in INSECT_GROUPS]

print(f"{len(frame)} images | {columns.summary()}")
frame[GROUP_COLS].sum().rename("n images").to_frame().T

In [ ]:
# Prevalence per measurement: used by the safeguard and to read the accuracies.
prevalence = []
for measure in MEASUREMENTS:
    status = f"{measure}{STATUS_SUFFIX}"
    if status not in frame.columns:
        continue
    y = 1 - frame[status].astype(int).to_numpy()   # 1 = non measurable
    prevalence.append({
        "measure": measure,
        "n": len(y),
        "n_unmeasurable": int(y.sum()),
        "prevalence_unmeasurable": float(y.mean()),
        "kept": bool(min(y.sum(), len(y) - y.sum()) >= MIN_MINORITY),
    })

prevalence = pd.DataFrame(prevalence)
prevalence.to_csv(OUT_DIR / "prevalence.csv", index=False)

KEPT = prevalence.loc[prevalence["kept"], "measure"].tolist()
SKIPPED = prevalence.loc[~prevalence["kept"], "measure"].tolist()
print(f"{len(KEPT)} measurements kept, {len(SKIPPED)} skipped (< {MIN_MINORITY} minority examples)")
print("skipped:", SKIPPED)
prevalence

## 3. Feature sets and approaches

In [ ]:
def conf_columns(points) -> list:
    """Existing confidence columns for a list of keypoints."""
    return [columns.conf[p] for p in points if p in columns.conf]


ALL_CONF = conf_columns(POINTS)


def feature_sets(measure: str) -> dict:
    return {
        "direct": conf_columns(MEAS_TO_KP[measure]),
        "related": conf_columns(related_entities(measure)[0]),
        "all": ALL_CONF,
    }


def target(measure: str) -> np.ndarray:
    """1 = non measurable (positive, minority class)."""
    return 1 - frame[f"{measure}{STATUS_SUFFIX}"].astype(int).to_numpy()


def make_xgb(y_train: np.ndarray) -> XGBClassifier:
    n_pos = max(int(y_train.sum()), 1)
    n_neg = max(len(y_train) - n_pos, 1)
    return XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        scale_pos_weight=n_neg / n_pos,          # imbalance
        eval_metric="logloss", tree_method="hist",
        n_jobs=-1, random_state=RANDOM_STATE,
    )


def make_rf(y_train: np.ndarray) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=200, min_samples_leaf=2,
        class_weight="balanced",                  # imbalance
        n_jobs=-1, random_state=RANDOM_STATE,
    )


# (key, type, feature set, model factory)
APPROACHES = [
    #("rule_conf_mean", "rule", "direct", None),
    #("rule_conf_min", "rule", "direct", None),
    #("xgb_direct", "model", "direct", make_xgb),
    #("rf_direct", "model", "direct", make_rf),
    #("xgb_related", "model", "related", make_xgb),
    ("rf_related", "model", "related", make_rf),
    #("xgb_all", "model", "all", make_xgb),
    #("rf_all", "model", "all", make_rf),
]
NAMES = [a[0] for a in APPROACHES]

## 4. Score, threshold and cross-validation

Every approach produces a **score increasing with the probability of "non measurable"**,
which makes PR curves and confusion matrices directly comparable.
An undetected keypoint (empty cell) counts as a zero confidence for the threshold rules,
stays `NaN` for XGBoost (native informative branch) and is imputed to `-1` for the random forest.

In [ ]:
def rule_score(values: pd.DataFrame, how: str) -> np.ndarray:
    data = np.nan_to_num(values.to_numpy(dtype=float), nan=0.0)  # missing kp -> conf 0
    agg = data.mean(axis=1) if how == "mean" else data.min(axis=1)
    return -agg   # low confidence -> high score -> non measurable


def best_threshold(y: np.ndarray, score: np.ndarray) -> float:
    """Threshold maximising the MCC, searched on data not seen during training."""
    candidates = np.unique(np.quantile(score, np.linspace(0.0, 1.0, 201)))
    best_t, best_m = candidates[0], -np.inf
    for t in candidates:
        m = matthews_corrcoef(y, (score >= t).astype(int))
        if m > best_m:
            best_t, best_m = t, m
    return float(best_t)


def evaluate_measure(measure: str) -> dict:
    """Out-of-fold predictions of the 8 approaches for one measurement."""
    y = target(measure)
    sets = feature_sets(measure)
    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    scores = {name: np.zeros(len(y)) for name in NAMES}
    preds = {name: np.zeros(len(y), dtype=int) for name in NAMES}
    thresholds = {name: [] for name in NAMES}

    for train_idx, test_idx in splitter.split(np.zeros(len(y)), y):
        # inner partition: only used to set the decision threshold
        inner_fit, inner_val = train_test_split(
            train_idx, test_size=0.25, stratify=y[train_idx], random_state=RANDOM_STATE
        )
        for name, kind, which, factory in APPROACHES:
            cols = sets[which]
            if not cols:
                continue
            if kind == "rule":
                how = "mean" if name.endswith("mean") else "min"
                val_score = rule_score(frame.iloc[inner_val][cols], how)
                test_score = rule_score(frame.iloc[test_idx][cols], how)
            else:
                data = frame[cols + GROUP_COLS]
                if name.startswith("rf"):
                    data = data.fillna(NA_FILL)   # the RF does not handle NaN
                model = factory(y[inner_fit])
                model.fit(data.iloc[inner_fit], y[inner_fit])
                if name == "rf_related":
                    joblib.dump(model, f"models/rf_related_{measure}.joblib")
                val_score = model.predict_proba(data.iloc[inner_val])[:, 1]
                del model
                gc.collect()
                model = factory(y[train_idx])
                model.fit(data.iloc[train_idx], y[train_idx])
                test_score = model.predict_proba(data.iloc[test_idx])[:, 1]
                del model
                gc.collect()

            threshold = best_threshold(y[inner_val], val_score)
            scores[name][test_idx] = test_score
            preds[name][test_idx] = (test_score >= threshold).astype(int)
            thresholds[name].append(threshold)

    return {"y": y, "scores": scores, "preds": preds, "thresholds": thresholds}

## 5. Metrics and figures

In [ ]:
def metric_rows(measure: str, result: dict) -> list:
    y = result["y"]
    prev = float(y.mean())
    rows = []
    for name in NAMES:
        score, pred = result["scores"][name], result["preds"][name]
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        ap = float(average_precision_score(y, score))
        rows.append({
            "measure": measure,
            "model": name,
            "n": len(y),
            "prevalence_unmeasurable": prev,
            "mcc": float(matthews_corrcoef(y, pred)),
            "average_precision": ap,
            "average_precision_norm": (ap - prev) / (1 - prev) if prev < 1 else np.nan,
            "accuracy": float((tp + tn) / len(y)),
            "accuracy_unmeasurable": float(tp / (tp + fn)) if (tp + fn) else np.nan,
            "accuracy_measurable": float(tn / (tn + fp)) if (tn + fp) else np.nan,
            "balanced_accuracy": float(0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))),
            "precision_unmeasurable": float(tp / (tp + fp)) if (tp + fp) else np.nan,
            "threshold_median": float(np.median(result["thresholds"][name])),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        })
    return rows


def plot_pr(measure: str, result: dict) -> None:
    y = result["y"]
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for name in NAMES:
        score = result["scores"][name]
        precision, recall, _ = precision_recall_curve(y, score)
        ap = average_precision_score(y, score)
        ax.plot(recall, precision, lw=1.6, label=f"{name} (AP={ap:.3f})")
    ax.axhline(y.mean(), color="grey", ls="--", lw=1, label=f"chance ({y.mean():.3f})")
    ax.set_xlabel("Recall (non measurable)")
    ax.set_ylabel("Precision (non measurable)")
    ax.set_title(f"Precision-recall curve — {measure}")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "pr_curves" / f"{slug(measure)}.png", dpi=140)
    plt.close(fig)


def plot_confusion(measure: str, result: dict) -> None:
    y = result["y"]
    labels = ["measurable", "non meas."]
    fig, axes = plt.subplots(2, 4, figsize=(13, 7))
    for ax, name in zip(axes.ravel(), NAMES):
        matrix = confusion_matrix(y, result["preds"][name], labels=[0, 1])
        normalised = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)
        ax.imshow(normalised, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{matrix[i, j]}\n{normalised[i, j]:.0%}",
                        ha="center", va="center", fontsize=9,
                        color="white" if normalised[i, j] > 0.5 else "black")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_title(f"{name}\nMCC={matthews_corrcoef(y, result['preds'][name]):.3f}", fontsize=9)
        ax.set_xlabel("predicted", fontsize=8)
        ax.set_ylabel("actual", fontsize=8)
    fig.suptitle(f"Confusion matrices (out-of-fold) — {measure}")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "confusion" / f"{slug(measure)}.png", dpi=140)
    plt.close(fig)

## 6. Main loop

In [ ]:
rows, OOF = [], {}                        # <- instead of rows = []
for i, measure in enumerate(KEPT, start=1):
    print(f"[{i:2d}/{len(KEPT)}] {measure}", flush=True)
    result = evaluate_measure(measure)
    rows.extend(metric_rows(measure, result))
    plot_pr(measure, result)
    plot_confusion(measure, result)
    OOF[measure] = {"y": result["y"], **{n: result["preds"][n] for n in NAMES}}   # <- added
    del result
    gc.collect()

metrics = pd.DataFrame(rows)
metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
print(f"\n{len(metrics)} rows written to {OUT_DIR / 'metrics.csv'}")
metrics.head(8)

## 7. Approach comparison

Ranking by **mean MCC rank** over all the kept measurements
(rank 1 = best approach for the measurement), plus the number of wins.

In [ ]:
mcc = metrics.pivot(index="measure", columns="model", values="mcc")[NAMES]
ranks = mcc.rank(axis=1, ascending=False)

ranking = pd.DataFrame({
    "mean_rank": ranks.mean(),
    "median_mcc": mcc.median(),
    "mean_mcc": mcc.mean(),
    "mean_ap_norm": metrics.pivot(index="measure", columns="model",
                                  values="average_precision_norm")[NAMES].mean(),
    "mean_accuracy_unmeasurable": metrics.pivot(index="measure", columns="model",
                                                values="accuracy_unmeasurable")[NAMES].mean(),
    "wins": mcc.idxmax(axis=1).value_counts().reindex(NAMES).fillna(0).astype(int),
}).sort_values("mean_rank")

ranking.to_csv(OUT_DIR / "ranking.csv")
ranking

In [ ]:
# Measurement x approach heatmap (MCC)
fig, ax = plt.subplots(figsize=(9, 0.42 * len(mcc) + 2.5))
image = ax.imshow(mcc.to_numpy(), cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(NAMES)))
ax.set_yticks(range(len(mcc)))
ax.set_xticklabels(NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(mcc.index, fontsize=8)
for i in range(mcc.shape[0]):
    for j in range(mcc.shape[1]):
        value = mcc.iat[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if value < 0.6 else "black")
fig.colorbar(image, ax=ax, label="MCC")
ax.set_title("Out-of-fold MCC per measure and per approach")
fig.tight_layout()
fig.savefig(OUT_DIR / "heatmap_mcc.png", dpi=140)
plt.close(fig)

# Mean rank barplot
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(ranking.index[::-1], ranking["mean_rank"][::-1], color="steelblue")
for name, value in zip(ranking.index[::-1], ranking["mean_rank"][::-1]):
    ax.text(value + 0.05, name, f"{value:.2f}", va="center", fontsize=8)
ax.set_xlabel(f"MCC mean rank on {len(mcc)} measures (1 = best)")
ax.set_title("Approach ranking")
fig.tight_layout()
fig.savefig(OUT_DIR / "ranking.png", dpi=140)
plt.close(fig)

print("Figures written:", len(list(OUT_DIR.rglob('*.png'))))

In [ ]:
# --- Summary table: MCC, precision, recall ("non measurable" class) ---
summary = (
    metrics
    .rename(columns={
        "precision_unmeasurable": "precision",
        "accuracy_unmeasurable": "recall",
    })
    .groupby("model")[["mcc", "precision", "recall"]]
    .agg(["mean", "std"])
    .reindex(NAMES)
    .sort_values(("mcc", "mean"), ascending=False)
    .round(3)
)
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary.insert(0, "n_measures", metrics.groupby("model")["measure"].nunique().reindex(summary.index))

summary.to_csv(OUT_DIR / "summary_mcc_precision_recall.csv")
summary

In [ ]:
# --- Practical consequence: usable measurements per image -----------------
BASELINE = "rule_conf_min"           # "before": threshold on the minimum confidence
BEST = ranking.index[1]              # "after": Occam's razor: prefer the simplest model at equal performance
print(f"before = {BASELINE} | after = {BEST}")

y_true = np.column_stack([OOF[m]["y"] for m in KEPT])        # 1 = non measurable
valid = y_true == 0                                          # measurement actually usable

rows, per_image = [], {"image_name": frame["image_name"], "group": frame["group"].astype(str)}


def summarise(label, rejected):
    kept = ~rejected
    kept_valid = kept & valid
    per_image[f"n_kept_{label}"] = kept.sum(axis=1)
    per_image[f"n_valid_kept_{label}"] = kept_valid.sum(axis=1)
    rows.append({
        "filtering": label,
        "kept_measures_per_image": kept.sum(axis=1).mean(),
        "of_which_actually_valid": kept_valid.sum(axis=1).mean(),
        "valid_lost_per_image": (valid & rejected).sum(axis=1).mean(),
        "valid_retention": kept_valid.sum() / valid.sum(),
        "kept_contamination": (kept & ~valid).sum() / max(kept.sum(), 1),
    })


summarise("no_filtering", np.zeros_like(y_true, dtype=bool))
for name in (BASELINE, BEST):
    predicted = np.column_stack([OOF[m][name] for m in KEPT])
    summarise(name, predicted == 1)

practical = pd.DataFrame(rows).set_index("filtering").round(3)
practical.to_csv(OUT_DIR / "filtering_impact.csv")
pd.DataFrame(per_image).to_csv(OUT_DIR / "measures_per_image.csv", index=False)

print(f"\n{len(KEPT)} measurements evaluated, {valid.mean():.1%} actually usable\n")
practical

In [ ]:
# --- Number of input variables per approach --------------------------------
counts = []
for measure in KEPT:
    sets = feature_sets(measure)
    for name, kind, which, _ in APPROACHES:
        counts.append({
            "measure": measure,
            "model": name,
            "n_features": len(sets[which]) + (len(GROUP_COLS) if kind == "model" else 0),
        })

n_features = (
    pd.DataFrame(counts)
    .groupby("model")["n_features"]
    .agg(mean_n_features="mean", min_n_features="min", max_n_features="max")
    .reindex(NAMES)
    .round(2)
)
n_features["mean_mcc"] = metrics.groupby("model")["mcc"].mean().round(3)
n_features = n_features.sort_values("mean_n_features")

n_features.to_csv(OUT_DIR / "n_features.csv")
n_features


## 8. Additional figures

In [ ]:
# --- Label distribution per measurement (stacked bars) --------------------
order = prevalence.sort_values("prevalence_unmeasurable")
n_ok = (order["n"] - order["n_unmeasurable"]).to_numpy()
n_ko = order["n_unmeasurable"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 0.34 * len(order) + 2))
ax.barh(order["measure"], n_ok, color="#4c9f70", label="measurable")
ax.barh(order["measure"], n_ko, left=n_ok, color="#c0504d", label="non measurable")
for i, (a, b) in enumerate(zip(n_ok, n_ko)):
    ax.text(a + b + order["n"].max() * 0.01, i, f"{b / (a + b):.0%}", va="center", fontsize=7)
ax.set_xlim(0, order["n"].max() * 1.09)   # margin for the percentage labels
ax.set_xlabel("number of images")
ax.set_title("Label repartition per measure (sorted by non-measurable ratio)")
ax.tick_params(labelsize=8)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(OUT_DIR / "labels_per_measure.png", dpi=140)
plt.close(fig)
print("written:", OUT_DIR / "labels_per_measure.png")

In [ ]:
# --- MCC per measurement: xgb_related vs rf_related ------------------------
PAIR = ["xgb_related", "rf_related"]
pair_mcc = mcc[PAIR].sort_values("xgb_related")
positions = np.arange(len(pair_mcc))
height = 0.38

fig, ax = plt.subplots(figsize=(9, 0.42 * len(pair_mcc) + 2))
ax.barh(positions + height / 2, pair_mcc["xgb_related"], height=height,
        color="#4472c4", label="xgb_related")
ax.barh(positions - height / 2, pair_mcc["rf_related"], height=height,
        color="#ed7d31", label="rf_related")
ax.set_yticks(positions)
ax.set_yticklabels(pair_mcc.index, fontsize=8)
ax.set_xlim(min(0.0, float(pair_mcc.min().min()) - 0.05), 1.0)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("MCC (out-of-fold)")
ax.set_title("MCC per measure: models trained on all keypoints and measures")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(OUT_DIR / "mcc_xgb_related_vs_rf_related.png", dpi=140)
plt.close(fig)
print("written:", OUT_DIR / "mcc_xgb_related_vs_rf_related.png")

### Feature importance

The best-ranked model (excluding the threshold rules, which have no variables) is retrained
on the whole data, measurement by measurement. These importances are **descriptive**:
they show *which keypoints carry the signal*, they do not measure a performance.
Two classic caveats: impurity-based importances (random forest) favour high-cardinality
variables, and two correlated confidences share the credit.

In [ ]:
MODEL_APPROACHES = {a[0]: (a[2], a[3]) for a in APPROACHES if a[1] == "model"}
BEST_MODEL = next(name for name in ranking.index if name in MODEL_APPROACHES)
BEST_MODEL = BEST
WHICH, FACTORY = MODEL_APPROACHES[BEST_MODEL]
print(f"Best model: {BEST_MODEL} (features: {WHICH})")

(OUT_DIR / "importance").mkdir(exist_ok=True)
CONF_TO_POINT = {column: point for point, column in columns.conf.items()}


def pretty(column: str) -> str:
    return CONF_TO_POINT.get(column, column.replace("_one_hot", ""))


series = []
for measure in KEPT:
    cols = feature_sets(measure)[WHICH]
    if not cols:
        continue
    y = target(measure)
    data = frame[cols + GROUP_COLS]
    if BEST_MODEL.startswith("rf"):
        data = data.fillna(NA_FILL)

    model = FACTORY(y)
    model.fit(data, y)
    values = pd.Series(model.feature_importances_, index=[pretty(c) for c in data.columns])
    del model
    gc.collect()
    series.append(values.rename(measure))

    top = values.sort_values().tail(15)
    fig, ax = plt.subplots(figsize=(6.5, 0.32 * len(top) + 1.6))
    ax.barh(top.index, top.to_numpy(), color="steelblue")
    ax.set_xlabel("importance")
    ax.set_title(f"{BEST_MODEL}: {measure}", fontsize=10)
    ax.tick_params(labelsize=8)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "importance" / f"{slug(measure)}.png", dpi=140)
    plt.close(fig)

importance = pd.concat(series, axis=1).T
importance.to_csv(OUT_DIR / "feature_importance.csv")
print(f"{len(importance)} measurements, {importance.shape[1]} variables")

In [ ]:
# Mean importance over all the measurements
mean_importance = importance.mean().sort_values().tail(20)
coverage = importance.notna().sum()

fig, ax = plt.subplots(figsize=(7, 0.32 * len(mean_importance) + 2))
ax.barh(mean_importance.index, mean_importance.to_numpy(), color="#4472c4")
for i, name in enumerate(mean_importance.index):
    ax.text(mean_importance[name], i, f"  {coverage[name]}/{len(importance)}",
            va="center", fontsize=7, color="grey")
ax.set_xlim(0, float(mean_importance.max()) * 1.12)
ax.set_xlabel("mean importance (grey number: number of measures where the variable exists)")
ax.set_title(f"Most used variables ({BEST_MODEL})")
ax.tick_params(labelsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "mean_importance.png", dpi=140)
plt.close(fig)
print("written:", OUT_DIR / "mean_importance.png")

## 9. Reading the results

- `outputs/metrics.csv` — every metric, one row per (measurement, approach).
- `outputs/prevalence.csv` — counts and measurements left out by the safeguard.
- `outputs/ranking.csv` + `ranking.png` — global ranking of the 8 approaches.
- `outputs/heatmap_mcc.png` — which measurements are hard, independently of the approach.
- `outputs/pr_curves/*.png`, `outputs/confusion/*.png` — detail per measurement.
- `outputs/labels_per_measure.png` — measurable / non measurable distribution.
- `outputs/mcc_xgb_related_vs_rf_related.png` — direct comparison of the two related models.
- `outputs/importance/*.png`, `outputs/mean_importance.png`, `feature_importance.csv`.

Two points of attention for the interpretation:

1. `accuracy` must always be read against `1 - prevalence_unmeasurable` (accuracy of the
   constant classifier). An accuracy of 0.95 on a measurement with 5 % non-measurables is worthless.
2. The hind wing measurements are labelled non measurable *by construction* for the
   groups that have none: the MCC will be very high there simply because the group one-hot
   is enough. The table below isolates these cases.

In [ ]:
# Measurements whose label is almost determined by the taxonomic group.
suspects = []
for measure in KEPT:
    y = pd.Series(target(measure))
    by_group = y.groupby(frame["group"].astype(str).to_numpy()).mean()
    if ((by_group < 0.05) | (by_group > 0.95)).all():
        suspects.append({"measure": measure, **by_group.round(2).to_dict()})

suspects = pd.DataFrame(suspects)
if not suspects.empty:
    suspects.to_csv(OUT_DIR / "group_determined_measures.csv", index=False)
    print("Non-measurable rate per group (trivial measurements):")
suspects

In [ ]:
# save model

import joblib

joblib.dump(rf, "my_random_forest.joblib")